In [20]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [39]:
files = glob('WenetSpeech4TTS/Premium/WenetSpeech4TTS_Premium_*/txts/*.txt')
len(files)

244071

In [22]:
audio_df = pd.read_parquet('WenetSpeech4TTS_Premium.parquet')
audio = {}
for i in range(audio_df.shape[0]):
    audio[audio_df['audio'].iloc[i]] = i
len(audio)

252292

In [24]:
with open('group-WenetSpeech4TTS_Premium.json') as fopen:
    speakers = json.load(fopen)

In [74]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2]) + '_audio'
        os.makedirs(base, exist_ok=True)
        f_audio = f.replace('txts/', 'wavs/').replace('.txt', '.wav')
        if not os.path.exists(f_audio):
            continue
        
        with open(f) as fopen:
            t = fopen.read()
        splitted = t.split('\t')
        if len(t.split('\t')) != 3:
            continue
        t = splitted[1].strip()
        if len(t) < 2:
            continue

        f_new = f_audio.replace('/', '-').replace('.txt', '')
        audio_filename = f'{f_new}.mp3'
        audio_filename = os.path.join(base, audio_filename)
        audio_np, sr = sf.read(f_audio)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        speaker = speakers[str(audio[f_audio])]
        
        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': f"{base}_{speaker}".replace(' ', '_')
        })
        
    return data

In [76]:
data = loop((files[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 24.87it/s]


In [77]:
data = multiprocessing(files, loop, cores = 30)

100%|██████████| 8135/8135 [08:24<00:00, 16.12it/s]


In [78]:
len(data)

244071

In [79]:
data[0]

{'audio_filename': 'WenetSpeech4TTS_Premium_audio/WenetSpeech4TTS-Premium-WenetSpeech4TTS_Premium_1-wavs-X0000004352_8003104_S00152-S00154.wav.mp3',
 'text': '觉得上半年确实有一部分人通过炒股赚了首付，但是后来的结果就是余款付不出来了。唐老师来听您的，认为雷区还在。',
 'speaker': 'WenetSpeech4TTS_Premium_audio_speaker_0'}

In [80]:
data[-1]

{'audio_filename': 'WenetSpeech4TTS_Premium_audio/WenetSpeech4TTS-Premium-WenetSpeech4TTS_Premium_3-wavs-X0000014625_237106494_S00255-S00259.wav.mp3',
 'text': '爱情的水晶很美好，也很脆弱，需要双方细心用心的呵护，太沉重的负担和压力，只会让他尽早破碎。',
 'speaker': 'WenetSpeech4TTS_Premium_audio_speaker_163'}

In [81]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'WenetSpeech4TTS_Premium_audio/WenetSpeech4TTS-Premium-WenetSpeech4TTS_Premium_1-wavs-X0000004352_8003104_S00152-S00154.wav.mp3',
 'text': '觉得上半年确实有一部分人通过炒股赚了首付，但是后来的结果就是余款付不出来了。唐老师来听您的，认为雷区还在。',
 'speaker': 'WenetSpeech4TTS_Premium_audio_speaker_0'}

In [82]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'WenetSpeech4TTS_Premium')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  7.28ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  27%|██▋       | 7.88MB / 29.0MB,  895kB/s  
Processing Files (0 / 1): 100%|█████████▉| 28.9MB / 29.0MB, 3.21MB/s  
Processing Files (1 / 1): 100%|██████████| 29.0MB / 29.0MB, 3.09MB/s  
Processing Files (1 / 1): 100%|██████████| 29.0MB / 29.0MB, 3.02MB/s  
New Data Upload: 100%|██████████| 29.0MB / 29.0MB, 3.02MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.32s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/de0b34fdda36b648fa0fb69cf800e1f5815a5be2', commit_message='Upload dataset', commit_description='', oid='de0b34fdda36b648fa0fb69cf800e1f5815a5be2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [83]:
audio_files = [d['audio_filename'] for d in data]

with open('WenetSpeech4TTS_Premium-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [84]:
!du -hs WenetSpeech4TTS_Premium_audio

11G	WenetSpeech4TTS_Premium_audio


In [87]:
# !zip -rq WenetSpeech4TTS_Premium_audio.zip WenetSpeech4TTS_Premium_audio
# !hf upload malaysia-ai/Multilingual-TTS WenetSpeech4TTS_Premium_audio.zip --repo-type=dataset

In [90]:
# !zip -rq WenetSpeech4TTS_Premium_audio_neucodec.zip WenetSpeech4TTS_Premium_audio_neucodec

In [91]:
# !hf upload malaysia-ai/Multilingual-TTS WenetSpeech4TTS_Premium_audio_neucodec.zip --repo-type=dataset